# Entrenamiento U-Net Básico

En este notebook se implementa una primera arquitectura U-Net desarrollada manualmente en PyTorch para la detección de cambios en imágenes satelitales Sentinel-2.

El objetivo de esta etapa es construir una línea base inicial para segmentación semántica de áreas afectadas por deforestación.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
tiles_path1 = "/content/drive/MyDrive/vision_sentinental_drive/tiles_par1"
labels_path1 = "/content/drive/MyDrive/vision_sentinental_drive/labels_par1"

In [3]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from torch.utils.data import random_split

In [4]:
class ChangeDetectionDataset(Dataset):
    def __init__(self, tiles_dir, labels_dir):
        self.tiles_dir = tiles_dir
        self.labels_dir = labels_dir

        # obtener IDs únicos (0,1,2,...)
        self.ids = sorted([
            f.split("_")[1]
            for f in os.listdir(tiles_dir)
            if "pre" in f
        ])

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        tile_id = self.ids[idx]

        pre = np.load(os.path.join(self.tiles_dir, f"tile_{tile_id}_pre.npy"))
        post = np.load(os.path.join(self.tiles_dir, f"tile_{tile_id}_post.npy"))

        # concatenar
        x = np.concatenate([pre, post], axis=0)

        label = np.load(os.path.join(self.labels_dir, f"tile_{tile_id}_label.npy"))
        label = np.expand_dims(label, axis=0)

        x = torch.tensor(x, dtype=torch.float32)
        label = torch.tensor(label, dtype=torch.float32)

        return x, label

In [5]:
dataset = ChangeDetectionDataset(
    tiles_path1,
    labels_path1
)

print("Número de samples:", len(dataset))

x, y = dataset[0]

print("Input shape:", x.shape)
print("Label shape:", y.shape)

Número de samples: 16
Input shape: torch.Size([18, 512, 512])
Label shape: torch.Size([1, 512, 512])


In [6]:
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(
    dataset,
    [train_size, val_size]
)

print("Train:", len(train_dataset))
print("Val:", len(val_dataset))

Train: 12
Val: 4


In [7]:
train_loader = DataLoader(
    train_dataset,
    batch_size=2,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=2,
    shuffle=False
)

In [8]:
class UNet(nn.Module):
    def __init__(self, in_channels=18, out_channels=1):
        super(UNet, self).__init__()

        def conv_block(in_c, out_c):
            return nn.Sequential(
                nn.Conv2d(in_c, out_c, 3, padding=1),
                nn.ReLU(inplace=True),
                nn.Conv2d(out_c, out_c, 3, padding=1),
                nn.ReLU(inplace=True)
            )

        # Encoder
        self.enc1 = conv_block(in_channels, 64)
        self.pool1 = nn.MaxPool2d(2)

        self.enc2 = conv_block(64, 128)
        self.pool2 = nn.MaxPool2d(2)

        self.enc3 = conv_block(128, 256)
        self.pool3 = nn.MaxPool2d(2)

        # Bottleneck
        self.bottleneck = conv_block(256, 512)

        # Decoder
        self.up3 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.dec3 = conv_block(512, 256)

        self.up2 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec2 = conv_block(256, 128)

        self.up1 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec1 = conv_block(128, 64)

        # Output
        self.final = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        # Encoder
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))

        # Bottleneck
        b = self.bottleneck(self.pool3(e3))

        # Decoder
        d3 = self.up3(b)
        d3 = torch.cat([d3, e3], dim=1)
        d3 = self.dec3(d3)

        d2 = self.up2(d3)
        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.dec2(d2)

        d1 = self.up1(d2)
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1)

        return self.final(d1)

In [9]:
model = UNet(
    in_channels=18,
    out_channels=1
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = model.to(device)

print(device)

cpu


In [10]:
pos_weight = torch.tensor([5.0]).to(device)

criterion = torch.nn.BCEWithLogitsLoss(
    pos_weight=pos_weight
)

In [11]:
optimizer = optim.Adam(
    model.parameters(),
    lr=1e-4
)

In [ ]:
num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    train_loss = 0

    for x, y in train_loader:
        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        y_pred = model(x)

        loss = criterion(y_pred, y)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    # VALIDACIÓN
    model.eval()
    val_loss = 0

    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(device)
            y = y.to(device)

            y_pred = model(x)
            loss = criterion(y_pred, y)

            val_loss += loss.item()

    val_loss /= len(val_loader)

    print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

In [ ]:
model.eval()

# tomar un batch de validación
x, y = next(iter(val_loader))

x = x.to(device)

with torch.no_grad():
    y_pred = model(x)
    y_pred = torch.sigmoid(y_pred)  # convertir a probabilidades

# pasar a CPU
x = x.cpu()
y = y.cpu()
y_pred = y_pred.cpu()

# visualizar primer ejemplo del batch
i = 0

plt.figure(figsize=(12,4))

# Label real
plt.subplot(1,3,1)
plt.imshow(y[i,0], cmap="gray")
plt.title("Ground Truth")

# Predicción (probabilidad)
plt.subplot(1,3,2)
plt.imshow(y_pred[i,0], cmap="gray")
plt.title("Predicción (prob)")

# Predicción binaria
plt.subplot(1,3,3)
plt.imshow((y_pred[i,0] > 0.5), cmap="gray")
plt.title("Predicción binaria")

plt.show()